# Phase 2: Data Harmonisation

This notebook demonstrates the complete data harmonisation pipeline:

1. **Load Phase 1 data** - Raw ingested data from 3 sources
2. **Apply harmonisation** - 5-step pipeline
3. **Validate outputs** - Schema, coverage, data quality
4. **Visualize transforms** - Before/after comparisons

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add workspace to path
workspace_root = Path.cwd().parent
sys.path.insert(0, str(workspace_root))

from src.harmonise.config import (
    DPLACE_RAW, DRH_RAW, SESHAT_RAW,
    DPLACE_HARMONISED, DRH_HARMONISED, SESHAT_HARMONISED,
    ACTIVE_SOURCES, HARMONISED_COLUMN_ORDER
)
from src.harmonise.harmonise_all import HarmonisationPipeline

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print(f"✓ Workspace root: {workspace_root}")
print(f"✓ Active sources: {ACTIVE_SOURCES}")

## 1. Load Phase 1 Data

In [ ]:
# Load Phase 1 data
phase1_data = {}

if Path(DPLACE_RAW).exists():
    phase1_data["dplace"] = pd.read_parquet(DPLACE_RAW)
    print(f"✓ D-PLACE: {len(phase1_data['dplace'])} records")

if Path(DRH_RAW).exists():
    phase1_data["drh"] = pd.read_parquet(DRH_RAW)
    print(f"✓ DRH: {len(phase1_data['drh'])} records")

if Path(SESHAT_RAW).exists():
    phase1_data["seshat"] = pd.read_parquet(SESHAT_RAW)
    print(f"✓ Seshat: {len(phase1_data['seshat'])} records")

print(f"\nPhase 1 columns: {list(phase1_data['dplace'].columns)}")

### Sample Phase 1 Data (D-PLACE)

In [ ]:
# Show sample Phase 1 data
phase1_data["dplace"].head(5).to_string()

## 2. Load Harmonised Data

In [ ]:
# Load harmonised data produced by harmonise_all.py
harmonised_data = {}

if Path(DPLACE_HARMONISED).exists():
    harmonised_data["dplace"] = pd.read_parquet(DPLACE_HARMONISED)
    print(f"✓ D-PLACE harmonised: {len(harmonised_data['dplace'])} records × {len(harmonised_data['dplace'].columns)} columns")

if Path(DRH_HARMONISED).exists():
    harmonised_data["drh"] = pd.read_parquet(DRH_HARMONISED)
    print(f"✓ DRH harmonised: {len(harmonised_data['drh'])} records × {len(harmonised_data['drh'].columns)} columns")

if Path(SESHAT_HARMONISED).exists():
    harmonised_data["seshat"] = pd.read_parquet(SESHAT_HARMONISED)
    print(f"✓ Seshat harmonised: {len(harmonised_data['seshat'])} records × {len(harmonised_data['seshat'].columns)} columns")

print(f"\nHarmonised columns: {list(harmonised_data['dplace'].columns)}")

### Sample Harmonised Data (D-PLACE)

In [ ]:
# Show sample harmonised data
harmonised_data["dplace"].head(5).to_string()

## 3. Before/After: Feature Mapping

In [ ]:
# Show how raw variables map to harmonised features
print("\nExample crosswalk mappings (D-PLACE):")

sample_rows = harmonised_data["dplace"][["variable_name", "variable_value", "feature_name", "feature_value"]].head(20)

# Group by variable to show unique mappings
print("\nVariable → Feature mappings:")
mappings = harmonised_data["dplace"][
    harmonised_data["dplace"]["feature_name"].notna()
].groupby("variable_name")["feature_name"].unique()

for var, features in mappings.head(10).items():
    print(f"  {var:20s} → {', '.join(features.astype(str))}")

## 4. Unit Standardisation

In [ ]:
# Unit type standardisation
print("\nUnit type standardisation (D-PLACE):")
print(harmonised_data["dplace"]["unit_type"].value_counts())

print("\nUnit ambiguity flags:")
ambiguous = harmonised_data["dplace"]["unit_ambiguous"].sum()
print(f"  Ambiguous units: {ambiguous} / {len(harmonised_data['dplace'])} ({100*ambiguous/len(harmonised_data['dplace']):.1f}%)")

# Show ambiguous unit notes
ambiguous_notes = harmonised_data["dplace"][harmonised_data["dplace"]["unit_ambiguous"] == 1]["unit_note"].unique()
if len(ambiguous_notes) > 0:
    print(f"\nAmbiguity notes:")
    for note in ambiguous_notes[:5]:
        print(f"  - {note}")

## 5. Temporal Standardisation

In [ ]:
# Temporal mode distribution
print("\nTemporal mode distribution:")
print(harmonised_data["dplace"]["temporal_mode"].value_counts())

print("\nTime uncertainty distribution:")
uncertainty_dist = harmonised_data["dplace"]["time_uncertainty"].value_counts().sort_index()
uncertainty_labels = {
    0: "Certain (≤10yr span)",
    1: "Low (≤100yr span)",
    2: "Medium (≤500yr span)",
    3: "High (>500yr span)"
}
for unc, count in uncertainty_dist.items():
    print(f"  {unc}: {uncertainty_labels[unc]:30s} - {count:5d} records ({100*count/len(harmonised_data['dplace']):5.1f}%)")

### Temporal Distribution Visualization

In [ ]:
# Create temporal visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Temporal mode
harmonised_data["dplace"]["temporal_mode"].value_counts().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Temporal Mode Distribution (D-PLACE)")
axes[0].set_ylabel("Number of records")
axes[0].set_xlabel("Temporal mode")

# Time uncertainty
time_unc = harmonised_data["dplace"]["time_uncertainty"].value_counts().sort_index()
time_unc.plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("Time Uncertainty Distribution (D-PLACE)")
axes[1].set_ylabel("Number of records")
axes[1].set_xlabel("Uncertainty level (0=certain, 3=high)")

plt.tight_layout()
plt.show()

## 6. Data Quality Scores

In [ ]:
# Data quality score statistics
print("Data Quality Score Statistics:")
print(harmonised_data["dplace"]["data_quality_score"].describe())

print(f"\nQuality score by source:")
for source, df in harmonised_data.items():
    if len(df) > 0:
        mean_score = df["data_quality_score"].mean()
        print(f"  {source:10s}: mean={mean_score:.3f}")

### Data Quality Visualization

In [ ]:
# Quality score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
harmonised_data["dplace"]["data_quality_score"].hist(bins=50, ax=axes[0], color="green", alpha=0.7)
axes[0].set_title("Data Quality Score Distribution (D-PLACE)")
axes[0].set_xlabel("Data Quality Score")
axes[0].set_ylabel("Frequency")

# By temporal mode
dplace_data = harmonised_data["dplace"]
dplace_data.boxplot(column="data_quality_score", by="temporal_mode", ax=axes[1])
axes[1].set_title("Data Quality Score by Temporal Mode (D-PLACE)")
axes[1].set_xlabel("Temporal Mode")
axes[1].set_ylabel("Data Quality Score")
plt.suptitle("")  # Remove default title

plt.tight_layout()
plt.show()

## 7. Coverage Audit

In [ ]:
# Load coverage audit
coverage_audit_path = Path("data/processed/harmonised/coverage_audit.csv")

if coverage_audit_path.exists():
    coverage_audit = pd.read_csv(coverage_audit_path)
    print(f"Coverage audit: {len(coverage_audit)} region×time_bin×source cells")
    print(f"\nGap severity distribution:")
    print(coverage_audit["gap_severity"].value_counts())
    
    print(f"\nGaps by source:")
    for source in coverage_audit["source"].unique():
        source_gaps = coverage_audit[coverage_audit["source"] == source]
        red_gaps = len(source_gaps[source_gaps["gap_severity"] == "RED"])
        yellow_gaps = len(source_gaps[source_gaps["gap_severity"] == "YELLOW"])
        print(f"  {source}: RED={red_gaps}, YELLOW={yellow_gaps}")
else:
    print("Coverage audit file not found")

### Gap Heatmap

In [ ]:
if coverage_audit_path.exists():
    # Create pivot table for visualization
    coverage_audit_data = pd.read_csv(coverage_audit_path)
    
    # Convert gap severity to numeric (for coloring)
    severity_map = {"GREEN": 0, "YELLOW": 1, "RED": 2}
    coverage_audit_data["severity_num"] = coverage_audit_data["gap_severity"].map(severity_map)
    
    # Pivot for D-PLACE
    dplace_coverage = coverage_audit_data[coverage_audit_data["source"] == "dplace"]
    if len(dplace_coverage) > 0:
        pivot = dplace_coverage.pivot_table(
            values="severity_num", 
            index="region", 
            columns="time_bin",
            aggfunc="first"
        )
        
        # Create heatmap
        fig, ax = plt.subplots(figsize=(14, 5))
        sns.heatmap(
            pivot, 
            cmap="RdYlGn_r", 
            cbar_kws={"label": "Gap Severity (0=GREEN, 1=YELLOW, 2=RED)"},
            ax=ax
        )
        ax.set_title("Coverage Gap Heatmap by Region & Time Bin (D-PLACE)")
        ax.set_xlabel("Time Bin")
        ax.set_ylabel("Region")
        plt.tight_layout()
        plt.show()

## 8. Schema Validation

In [ ]:
# Validate schema consistency across sources
print("Schema validation:")
print(f"\nExpected columns: {len(HARMONISED_COLUMN_ORDER)}")

for source, df in harmonised_data.items():
    actual_cols = set(df.columns)
    expected_cols = set(HARMONISED_COLUMN_ORDER)
    
    missing = expected_cols - actual_cols
    extra = actual_cols - expected_cols
    
    print(f"\n{source.upper()}:")
    print(f"  Actual columns: {len(actual_cols)}")
    if missing:
        print(f"  Missing: {missing}")
    if extra:
        print(f"  Extra: {extra}")
    if not missing and not extra:
        print(f"  ✓ Schema valid")

## 9. Data Integrity Checks

In [ ]:
# Run data integrity checks
print("Data Integrity Checks:")

for source, df in harmonised_data.items():
    print(f"\n{source.upper()}:")
    
    # Check for required columns non-null
    required_cols = ["source", "culture_id", "culture_name", "unit_type"]
    for col in required_cols:
        nulls = df[col].isna().sum()
        status = "✓" if nulls == 0 else "✗"
        print(f"  {status} {col}: {nulls} NAs")
    
    # Check value ranges
    if "lat" in df.columns:
        lat_valid = ((df["lat"] >= -90) & (df["lat"] <= 90)).sum()
        print(f"  ✓ Latitude valid: {lat_valid}/{len(df)}")
    
    # Check data quality scores
    score_min = df["data_quality_score"].min()
    score_max = df["data_quality_score"].max()
    print(f"  ✓ Data quality score range: [{score_min:.3f}, {score_max:.3f}]")
    
    # Check no duplicates within source
    dupes = df.duplicated().sum()
    print(f"  {'✓' if dupes == 0 else '✗'} Duplicate rows: {dupes}")

## 10. Summary

In [ ]:
print("\n" + "="*80)
print("HARMONISATION PIPELINE SUMMARY")
print("="*80)

print("\nPhase 1 → Phase 2 Transformation:")
for source in ACTIVE_SOURCES:
    if source in phase1_data and source in harmonised_data:
        phase1_rows = len(phase1_data[source])
        phase1_cols = len(phase1_data[source].columns)
        harm_rows = len(harmonised_data[source])
        harm_cols = len(harmonised_data[source].columns)
        
        print(f"\n{source.upper()}:")
        print(f"  Phase 1:      {phase1_rows:6d} rows × {phase1_cols:2d} columns")
        print(f"  Harmonised:   {harm_rows:6d} rows × {harm_cols:2d} columns")
        print(f"  Delta:        {harm_rows - phase1_rows:+6d} rows")

print("\n" + "="*80)
print("✓ VALIDATION COMPLETE")
print("="*80)